## Cell 1 — Imports & Setup

In [18]:
import pandas as pd
import numpy as np
import json
import re
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.linear_model import LinearRegression
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cpu"
print(f"Using device: {device}")

Using device: cpu


## Cell 2 — Load Data

In [19]:
df = pd.read_csv("data/customer_support_data.csv")
print(df.shape)
print(df.columns.tolist())
print(df["Priority_Level"].value_counts())
df.head()

(20000, 12)
['Ticket_ID', 'Customer_Name', 'Customer_Email', 'Ticket_Subject', 'Ticket_Description', 'Issue_Category', 'Priority_Level', 'Ticket_Channel', 'Submission_Date', 'Resolution_Time_Hours', 'Assigned_Agent', 'Satisfaction_Score']
Priority_Level
Low         7716
Medium      7570
High        3416
Critical    1298
Name: count, dtype: int64


,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,Assigned_Agent,Satisfaction_Score
0,TKT-100000,George Simon,lisastrickland@example.com,Hours of operation - Individual,"Hi Support, Where is your headquarters located...",General Inquiry,High,Web Form,2025-07-02,43,David Kim,5
1,TKT-100001,Scott Thompson,wevans@example.org,Data not syncing - Card,"Hi Support, The application crashes every time...",Technical,High,Chat,2025-06-28,41,Elena Rodriguez,5
2,TKT-100002,Jennifer Smith,oleonard@example.net,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...",Account,High,Web Form,2025-02-05,7,Anya Sharma,5
3,TKT-100003,Rachel Bullock,katherine67@example.net,Login failed - Let,"Hi Support, The dashboard is not loading any d...",Technical,Low,Web Form,2025-03-20,41,Anya Sharma,5
4,TKT-100004,Thomas Parks DDS,raykelsey@example.com,Refund status - Attention,"Hi Support, I have been trying to update my pa...",Billing,Medium,Email,2025-04-27,40,David Kim,5


## Cell 3 — Preprocessing

In [20]:
# Normalize column names
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

df = df.rename(columns={'priority_level': 'ticket_priority', 'issue_category': 'ticket_type'})

# Fill nulls
df["ticket_description"] = df["ticket_description"].fillna("")
df["ticket_subject"] = df["ticket_subject"].fillna("")
df["ticket_channel"] = df["ticket_channel"].fillna("unknown")
df["ticket_type"] = df["ticket_type"].fillna("unknown")
df["ticket_priority"] = df["ticket_priority"].str.strip().str.lower()

# Combine text fields
df["full_text"] = df["ticket_subject"] + " " + df["ticket_description"]

# Encode priority as numeric for comparison
priority_map = {"low": 0, "medium": 1, "high": 2, "critical": 3}
df["priority_numeric"] = df["ticket_priority"].map(priority_map)

# Parse resolution time to hours (assumes format like "2 days" or "48 hours")
def parse_resolution_time(val):
    if pd.isna(val):
        return np.nan
    val = str(val).lower()
    numbers = re.findall(r"[\d.]+", val)
    if not numbers:
        return np.nan
    num = float(numbers[0])
    if "day" in val:
        return num * 24
    return num

df["resolution_hours"] = df["resolution_time_hours"].apply(parse_resolution_time)
df["resolution_hours"] = df["resolution_hours"].fillna(df["resolution_hours"].median())

print(df[["ticket_priority", "priority_numeric", "resolution_hours"]].head())

  ticket_priority  priority_numeric  resolution_hours
0            high                 2              43.0
1            high                 2              41.0
2            high                 2               7.0
3             low                 0              41.0
4          medium                 1              40.0


## Cell 4 — Signal 1: Rule-Based NLP Severity Score

In [ ]:
CRISIS_KEYWORDS = [
    "urgent", "critical", "outage", "down", "breach", "broken", "emergency",
    "failure", "not working", "data loss", "cannot access", "system failure",
    "escalate", "immediately", "asap", "severe", "blocked", "production down"
]

LOW_KEYWORDS = [
    "question", "inquiry", "how to", "tutorial", "info", "wondering",
    "when will", "update me", "minor", "small issue", "curious"
]

def rule_based_score(text):
    text = text.lower()
    score = 0
    for kw in CRISIS_KEYWORDS:
        if kw in text:
            score += 1
    for kw in LOW_KEYWORDS:
        if kw in text:
            score -= 0.5
    # Negation penalty
    negations = len(re.findall(r"\bnot\b|\bno\b|\bnever\b|\bwithout\b", text))
    score -= negations * 0.3
    return score

df["rule_score"] = df["full_text"].apply(rule_based_score)

# Normalize to 0-3 scale
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 3))
df["rule_score_norm"] = scaler.fit_transform(df[["rule_score"]])
print(df["rule_score_norm"].describe())

Preparing batch input file...
Uploading file to Gemini File API...


## Cell 5 — Signal 2: Embedding-Based Clustering

In [ ]:
print("Loading sentence transformer...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Encoding tickets...")
embeddings = embedder.encode(df["full_text"].tolist(), batch_size=64, show_progress_bar=True)

# KMeans into 4 clusters (maps to Low/Med/High/Critical)
kmeans = KMeans(n_clusters=4, random_state=SEED, n_init=10)
df["cluster"] = kmeans.fit_predict(embeddings)

# Map clusters to severity by matching cluster center distances
# Align clusters to priority scale using resolution_hours as proxy
cluster_severity = df.groupby("cluster")["resolution_hours"].mean().sort_values()
cluster_to_severity = {cluster: rank for rank, cluster in enumerate(cluster_severity.index)}
df["cluster_severity"] = df["cluster"].map(cluster_to_severity)

print(df[["cluster", "cluster_severity"]].value_counts())

Loading sentence transformer...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5586.91it/s]


Encoding tickets...


Batches: 100%|██████████| 313/313 [00:22<00:00, 14.06it/s]
Python(8254) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


cluster  cluster_severity
1        0                   5425
3        1                   5416
2        2                   5181
0        3                   3978
Name: count, dtype: int64


## Cell 6 — Signal 3: Resolution Time Regression Severity

In [ ]:
# Use resolution hours as direct severity proxy (longer = likely higher severity)
rt_scaler = MinMaxScaler(feature_range=(0, 3))
df["resolution_severity"] = rt_scaler.fit_transform(df[["resolution_hours"]])
print(df["resolution_severity"].describe())

count    20000.000000
mean         0.963789
std          0.887947
min          0.000000
25%          0.252101
50%          0.655462
75%          1.436975
max          3.000000
Name: resolution_severity, dtype: float64


## Cell 7 — Fuse Signals → Inferred Severity

In [ ]:
from scipy.stats import zscore

PRIORITY_MAP = {"low": 0, "medium": 1, "high": 2, "critical": 3} 

# Z-score normalize each signal before fusion for better calibration
df["rule_z"] = zscore(df["rule_score_norm"])
df["cluster_z"] = zscore(df["cluster_severity"].astype(float))
df["resolution_z"] = zscore(df["resolution_severity"])

df["inferred_severity_score"] = (
    0.4 * df["rule_z"] +
    0.4 * df["cluster_z"] +
    0.2 * df["resolution_z"]
)

# Use percentile-based binning instead of fixed thresholds
p25 = df["inferred_severity_score"].quantile(0.25)
p50 = df["inferred_severity_score"].quantile(0.50)
p75 = df["inferred_severity_score"].quantile(0.75)

def score_to_label(score):
    if score <= p25:   return "low"
    elif score <= p50: return "medium"
    elif score <= p75: return "high"
    else:              return "critical"

df["inferred_severity"] = df["inferred_severity_score"].apply(score_to_label)
df["inferred_severity_numeric"] = df["inferred_severity"].map(PRIORITY_MAP)
print(df["inferred_severity"].value_counts())

inferred_severity
high        5102
low         5004
medium      4997
critical    4897
Name: count, dtype: int64


## Cell 8 — Generate Pseudo Mismatch Labels

In [ ]:
df["severity_delta"] = df["inferred_severity_numeric"] - df["priority_numeric"]

# Only flag high-confidence mismatches (delta >= 2 on ALL three signals)
df["rule_severity"] = pd.cut(df["rule_score_norm"], bins=4, labels=[0,1,2,3]).astype(float)
df["signal_consensus"] = (
    (df["rule_severity"] - df["priority_numeric"]).abs() +
    (df["cluster_severity"] - df["priority_numeric"]).abs() +
    (df["resolution_severity"].round() - df["priority_numeric"]).abs()
)

# Mismatch only when delta >= 2 AND at least 2 signals agree on the discrepancy
df["mismatch_label"] = (
    (df["severity_delta"].abs() >= 2) &
    (df["signal_consensus"] >= 3.0)
).astype(int)

def mismatch_type(delta):
    if delta >= 2:    return "Hidden Crisis"
    elif delta <= -2: return "False Alarm"
    return "Consistent"

df["mismatch_type"] = df["severity_delta"].apply(mismatch_type)
print(df["mismatch_label"].value_counts())
df.to_csv("data/pseudo_labeled.csv", index=False)

mismatch_label
0    11701
1     8299
Name: count, dtype: int64


## Cell 9 — Ablation: Individual Signal Agreement

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Compare each signal to the fused label
rule_label = df["rule_score_norm"].apply(score_to_label)
cluster_label = df["cluster_severity"].apply(score_to_label)
resolution_label = df["resolution_severity"].apply(score_to_label)

rule_agree = (rule_label == df["inferred_severity"]).mean()
cluster_agree = (cluster_label == df["inferred_severity"]).mean()
resolution_agree = (resolution_label == df["inferred_severity"]).mean()

print(f"Rule-based agreement with fused label:       {rule_agree:.3f}")
print(f"Embedding cluster agreement with fused label: {cluster_agree:.3f}")
print(f"Resolution time agreement with fused label:  {resolution_agree:.3f}")

# Pairwise signal agreement (required metric)
pairwise_rule_cluster = (rule_label == cluster_label).mean()
pairwise_rule_res = (rule_label == resolution_label).mean()
pairwise_cluster_res = (cluster_label == resolution_label).mean()

print(f"\nPairwise Rule vs Cluster:     {pairwise_rule_cluster:.3f}")
print(f"Pairwise Rule vs Resolution:  {pairwise_rule_res:.3f}")
print(f"Pairwise Cluster vs Resolution: {pairwise_cluster_res:.3f}")

Rule-based agreement with fused label:       0.245
Embedding cluster agreement with fused label: 0.256
Resolution time agreement with fused label:  0.270

Pairwise Rule vs Cluster:     0.729
Pairwise Rule vs Resolution:  0.622
Pairwise Cluster vs Resolution: 0.576


## Cell 10 — Prepare Dataset for Fine-Tuning

In [ ]:
df_model = df[["full_text", "ticket_channel", "resolution_hours", "mismatch_label"]].copy()
df_model = df_model.dropna()

# Encode channel
le = LabelEncoder()
df_model["channel_encoded"] = le.fit_transform(df_model["ticket_channel"])

# Normalize resolution hours
df_model["resolution_norm"] = MinMaxScaler().fit_transform(df_model[["resolution_hours"]])

# Append structured metadata to text (simple fusion strategy)
df_model["input_text"] = (
    df_model["full_text"] +
    " [CHANNEL] " + df_model["ticket_channel"] +
    " [RESTIME] " + df_model["resolution_norm"].round(2).astype(str)
)

# Train/val/test split
train_df, test_df = train_test_split(df_model, test_size=0.2, random_state=SEED, stratify=df_model["mismatch_label"])
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=SEED, stratify=train_df["mismatch_label"])

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(train_df["mismatch_label"].value_counts())

Train: 14400 | Val: 1600 | Test: 4000
mismatch_label
0    8425
1    5975
Name: count, dtype: int64


## Cell 11 — Tokenize

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["input_text"], truncation=True, padding=True, max_length=256)

train_ds = Dataset.from_pandas(train_df[["input_text", "mismatch_label"]].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[["input_text", "mismatch_label"]].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[["input_text", "mismatch_label"]].reset_index(drop=True))

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column("mismatch_label", "labels")
val_ds = val_ds.rename_column("mismatch_label", "labels")
test_ds = test_ds.rename_column("mismatch_label", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 4000/4000 [00:00<00:00, 28012.50 examples/s]


## Cell 12 — Load Model + Apply LoRA

In [ ]:
MODEL_NAME = "distilbert-base-uncased"  # back to clean base, no conflicting heads

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,           # kept increased from before
    lora_alpha=32,  # kept increased from before
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"]
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 14109.88it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 887,042 || all params: 67,842,052 || trainable%: 1.3075


## Cell 13 — Class Weights for Imbalance

In [ ]:
from torch.nn import CrossEntropyLoss

label_counts = train_df["mismatch_label"].value_counts().sort_index()
total = len(train_df)
class_weights = torch.tensor([total / (2 * c) for c in label_counts], dtype=torch.float)
print("Class weights:", class_weights)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = CrossEntropyLoss(weight=class_weights)(logits, labels)
        return (loss, outputs) if return_outputs else loss

Class weights: tensor([0.8546, 1.2050])


## Cell 14 — Train

from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="models/deberta_lora",
    num_train_epochs=8,                  # was 4
    per_device_train_batch_size=8,       # was 16, smaller = better generalization
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    warmup_ratio=0.15,                   # was 0.1
    weight_decay=0.02,                   # was 0.01
    learning_rate=3e-5,                  # add explicit LR
    seed=SEED,
    fp16=False,
    use_cpu=True
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

## Cell 15 — Evaluate on Test Set

In [ ]:
preds_output = trainer.predict(test_ds)
preds = np.argmax(preds_output.predictions, axis=1)
labels = preds_output.label_ids

print(classification_report(labels, preds, target_names=["Consistent", "Mismatch"]))
print(f"Accuracy: {accuracy_score(labels, preds):.4f}")
print(f"Macro F1: {f1_score(labels, preds, average='macro'):.4f}")

# Save metrics
metrics_out = classification_report(labels, preds, target_names=["Consistent", "Mismatch"], output_dict=True)
os.makedirs("outputs/metrics", exist_ok=True)
with open("outputs/metrics/test_metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)

              precision    recall  f1-score   support

  Consistent       0.92      0.68      0.78      2827
    Mismatch       0.53      0.86      0.65      1173

    accuracy                           0.73      4000
   macro avg       0.73      0.77      0.72      4000
weighted avg       0.81      0.73      0.75      4000

Accuracy: 0.7340
Macro F1: 0.7193


## Cell 16 — Evidence Dossier Generation

In [ ]:
def generate_dossier(row):
    assigned = row["ticket_priority"]
    inferred = row["inferred_severity"]
    assigned_num = priority_map.get(assigned, 1)
    inferred_num = priority_map.get(inferred, 1)
    delta = inferred_num - assigned_num

    # Extract keyword hits
    text = row["full_text"].lower()
    matched_keywords = [kw for kw in CRISIS_KEYWORDS if kw in text]
    keyword_weight = round(min(len(matched_keywords) / len(CRISIS_KEYWORDS), 1.0), 3)

    dossier = {
        "ticket_id": str(row.get("ticket_id", row.name)),
        "assigned_priority": assigned,
        "inferred_severity": inferred,
        "mismatch_type": "Hidden Crisis" if delta > 0 else "False Alarm",
        "severity_delta": str(delta),
        "feature_evidence": [
            {
                "signal": "keyword",
                "value": ", ".join(matched_keywords[:5]) if matched_keywords else "none",
                "weight": str(keyword_weight)
            },
            {
                "signal": "resolution_time",
                "value": str(round(row["resolution_hours"], 1)) + " hours",
                "interpretation": "Above median — suggests elevated complexity" if row["resolution_hours"] > df["resolution_hours"].median() else "Below median — routine resolution"
            },
            {
                "signal": "embedding_cluster",
                "value": f"Cluster {row['cluster']}",
                "weight": str(round(row["cluster_severity"] / 3, 3))
            }
        ],
        "constraint_analysis": (
            f"The ticket was assigned '{assigned}' but semantic analysis and keyword signals indicate "
            f"'{inferred}' severity. Resolution time of {round(row['resolution_hours'], 1)} hours and "
            f"cluster membership further support this discrepancy, resulting in a severity delta of {delta}."
        ),
        "confidence": str(round(abs(delta) / 3, 3))
    }
    return dossier

# Generate dossiers for all mismatched tickets
mismatched = df[df["mismatch_label"] == 1].copy()
os.makedirs("outputs/dossiers", exist_ok=True)

dossiers = []
for _, row in mismatched.iterrows():
    d = generate_dossier(row)
    dossiers.append(d)

with open("outputs/dossiers/all_dossiers.json", "w") as f:
    json.dump(dossiers, f, indent=2)

print(f"Generated {len(dossiers)} dossiers")
print(json.dumps(dossiers[0], indent=2))

Generated 15094 dossiers
{
  "ticket_id": "TKT-100001",
  "assigned_priority": "high",
  "inferred_severity": "low",
  "mismatch_type": "False Alarm",
  "severity_delta": "-2",
  "feature_evidence": [
    {
      "signal": "keyword",
      "value": "none",
      "weight": "0.0"
    },
    {
      "signal": "resolution_time",
      "value": "41.0 hours",
      "interpretation": "Above median \u2014 suggests elevated complexity"
    },
    {
      "signal": "embedding_cluster",
      "value": "Cluster 1",
      "weight": "0.0"
    }
  ],
  "constraint_analysis": "The ticket was assigned 'high' but semantic analysis and keyword signals indicate 'low' severity. Resolution time of 41.0 hours and cluster membership further support this discrepancy, resulting in a severity delta of -2.",
  "confidence": "0.667"
}
